# Classical Machine Learning Pipeline

## Workflow

1. Load the dataset
2. Split the dataset into training and testing sets
3. Convert text into TF-IDF features
4. Train the Logistic Regression model
5. Train the Naive Bayes model
6. Tune hyperparameters using GridSearchCV
7. Perform 5-fold Stratified Cross-Validation
8. Save the trained models using Pickle
9. Evaluate model performance
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - Confusion Matrix

In [4]:
pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 28.8 MB/s eta 0:00:000m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 36.0 MB/s eta 0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.2 MB/s eta 0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 34.2 MB/s eta 0:00:000m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 

Files removed: 566
Note: you may need to restart the kernel to use updated packages.


# Load the Dataset

In [1]:
import os
import sys
import pickle
import pandas as pd
import numpy as np
import mlflow

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

In [3]:
# Load the preprocessed dataset
df = pd.read_parquet("data/processed/reviews_unified.parquet")

# Remove rows with missing values
df = df.dropna(subset=["cleaned_text", "sentiment_label"])

# Remove very short reviews
df = df[df["cleaned_text"].str.len() > 5]

# Display dataset information
print(f"Loaded {len(df):,} reviews")
print("\nSentiment Distribution:")
print(df["sentiment_label"].value_counts())

Loaded 592,354 reviews

Sentiment Distribution:
sentiment_label
negative    218274
positive    209607
neutral     164473
Name: count, dtype: int64


In [4]:
# Display dataset shape
print(df.shape)

# Display column names
print(df.columns.tolist())

# Display the first five rows
df.head()

(592354, 8)
['platform', 'product_category', 'text', 'rating', 'date', 'review_id', 'sentiment_label', 'cleaned_text']


,platform,product_category,text,rating,date,review_id,sentiment_label,cleaned_text
0,amazon,All_Beauty,You get what you pay for. Sheds like there’s n...,1.0,2020-10-27,B07PLJJGDH_177807,negative,you get what you pay for. sheds like there s n...
1,amazon,All_Beauty,To transparent. Have to apply SEVERAL coats to...,2.0,2021-07-18,B08NJQ4Z22_145804,negative,to transparent. have to apply several coats to...
2,amazon,All_Beauty,Damage. Dislike received and it’s all broken,1.0,2019-06-05,B071J8QNTX_29248,negative,damage. dislike received and it s all broken
3,amazon,All_Beauty,Not Mink At All. Look very plasticy was not li...,1.0,2019-10-09,B07MSDRY4K_572462,negative,not mink at all. look very plasticy was not li...
4,amazon,All_Beauty,Just say no to alcohol denat. What I dislike i...,2.0,2018-06-14,B01MQD6ZM7_42199,negative,just say no to alcohol denat. what i dislike i...


## Encoding the Label

In [5]:
# Create a mapping from sentiment labels to numeric values
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

# Convert the sentiment labels into numeric labels
df["label_id"] = df["sentiment_label"].map(label_map)

# Remove rows where the sentiment label could not be mapped
df = df.dropna(subset=["label_id"])

# Convert the label column to integer type
df["label_id"] = df["label_id"].astype(int)



## Selecting Input and Target Features

In [6]:
# Store the cleaned review text as the input feature
X = df["cleaned_text"].tolist()

# Store the numeric sentiment labels as the target variable
y = df["label_id"].tolist()

# Train Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)
# Display the number of samples in each dataset
print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

Training samples : 473,883
Testing samples  : 118,471


# Convert text to TF-IDF

## Workflow

```text
Raw Reviews
      │
      ▼
Cleaned Reviews
      │
      ▼
TF-IDF Vectorisation
      │
      ▼
Numerical Feature Matrix
      │
      ▼
Logistic Regression / Naive Bayes
      │
      ▼
Sentiment Prediction
```

**In this project, TF-IDF is used as the feature engineering technique to convert cleaned customer reviews into numerical vectors suitable for training classical machine learning models.**

### Experiment - can change from 50000 to 75000 too 100000

In [8]:
max_features = 50000

In [9]:
# Initialise the TF-IDF vectoriser
vectorizer = TfidfVectorizer(

    # Keep the 50,000 most informative terms
    max_features=max_features,

    # Extract both single words (unigrams) and pairs of consecutive words (bigrams)
    ngram_range=(1, 2),

    # Apply logarithmic scaling to reduce the influence of very frequent words
    sublinear_tf=True,

    # Ignore terms that appear in fewer than two documents
    min_df=2,

    # Convert accented characters to their Unicode equivalents
    strip_accents="unicode",

    # Perform tokenisation at the word level
    analyzer="word",
)

# Learn the vocabulary from the training data and convert it into TF-IDF features
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the testing data using the same learned vocabulary
X_test_vec = vectorizer.transform(X_test)

# Display the size of the learned vocabulary
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")

Vocabulary size: 50,000


# TF-IDF Feature Extraction

## Why do we use TF-IDF?

Machine learning algorithms cannot process raw text directly. Therefore, the cleaned review text must be converted into numerical feature vectors before training the sentiment classification models.

The **Term Frequency–Inverse Document Frequency (TF-IDF)** technique is used to transform text into a numerical representation while assigning higher importance to informative words and lower importance to common words.

---

## How does TF-IDF work?

TF-IDF consists of two components:

### 1. Term Frequency (TF)

Term Frequency measures how often a word appears within a document.

**Example**

Review:

```text
good good battery
```

| Word | Frequency |
|------|----------:|
| good | 2 |
| battery | 1 |

The word **good** appears more frequently than **battery**, resulting in a higher TF value.

---

### 2. Inverse Document Frequency (IDF)

Inverse Document Frequency measures how important a word is across the entire dataset.

Common words such as **the**, **is**, and **and** appear in almost every review and therefore receive a lower weight.

Rare but informative words such as **excellent**, **terrible**, or **defective** receive higher weights because they provide more useful information for sentiment classification.

---

## Example

Consider three reviews:

| Review |
|---------|
| good battery life |
| bad battery life |
| excellent camera |

After applying TF-IDF, each review is converted into a numerical feature vector.

| Review | TF-IDF Representation (Illustrative) |
|---------|--------------------------------------|
| good battery life | [0.81, 0.42, 0.00, 0.00] |
| bad battery life | [0.00, 0.42, 0.76, 0.00] |
| excellent camera | [0.00, 0.00, 0.00, 0.95] |

These numerical vectors are then used as input for the machine learning models.

---

## Why is `fit_transform()` applied only to the training data?

```python
X_train_vec = vectorizer.fit_transform(X_train)
```

The `fit_transform()` method performs two operations:

1. Learns the vocabulary from the training dataset (`fit()`).
2. Converts the training text into TF-IDF feature vectors (`transform()`).

The testing dataset is processed using:

```python
X_test_vec = vectorizer.transform(X_test)
```

This ensures that the testing data is transformed using the vocabulary learned from the training data, preventing **data leakage** and enabling an unbiased evaluation of model performance.

---

## TF-IDF Parameters Used

| Parameter | Purpose |
|-----------|---------|
| `max_features=50000` | Retains the 50,000 most informative terms. |
| `ngram_range=(1,2)` | Extracts both unigrams (single words) and bigrams (two consecutive words). |
| `sublinear_tf=True` | Applies logarithmic scaling to reduce the influence of very frequent words. |
| `min_df=2` | Removes words appearing in fewer than two documents. |
| `strip_accents="unicode"` | Normalises accented characters to improve consistency. |
| `analyzer="word"` | Performs tokenisation at the word level. |

---



## GridSearchCV

In [10]:
# Search different values of the regularisation parameter (C)
lr_param_grid = {"C": [0.01, 0.1, 1.0, 5.0, 10.0]}

# Initialise GridSearchCV to find the best hyperparameter
lr_grid = GridSearchCV(

    # Logistic Regression classifier
  LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=42,
),

    # Hyperparameter values to evaluate
    lr_param_grid,

    # Perform 3-fold cross-validation
    cv=3,

    # Select the model with the highest Macro F1-score
    scoring="f1_macro",

    # Use all available CPU cores
    n_jobs=-1,

    # Suppress detailed training output
    verbose=0,
)

# Train the GridSearchCV model
lr_grid.fit(X_train_vec, y_train)

# Retrieve the best hyperparameter
best_lr_C = lr_grid.best_params_["C"]



In [11]:
# Display the best C value and corresponding Macro F1-score
print(f"Best C = {best_lr_C} (CV F1 macro = {lr_grid.best_score_:.4f})")

# Display the score for every C value tested
print(
    f"All scores: "
    f"{ {str(p['C']): round(s, 4) for p, s in zip(lr_grid.cv_results_['params'], lr_grid.cv_results_['mean_test_score'])} }"
)

Best C = 1.0 (CV F1 macro = 0.7920)
All scores: {'0.01': np.float64(0.7322), '0.1': np.float64(0.7792), '1.0': np.float64(0.792), '5.0': np.float64(0.7822), '10.0': np.float64(0.775)}


In [12]:
import sklearn
print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.0


In [13]:
from sklearn.linear_model import LogisticRegression

print(LogisticRegression)

<class 'sklearn.linear_model._logistic.LogisticRegression'>


# Hyperparameter Tuning Results

## Objective

The objective of this experiment was to identify the optimal value of the regularisation parameter (**C**) for the Logistic Regression classifier using **GridSearchCV**.

Five candidate values of **C** were evaluated:

```python
[0.01, 0.1, 1.0, 5.0, 10.0]
```

The model performance was assessed using **3-fold cross-validation**, with **Macro F1-score** selected as the evaluation metric.

---

## GridSearchCV Results

| C Value | Mean Macro F1-score |
|---------:|--------------------:|
| 0.01 | 0.7322 |
| 0.10 | 0.7792 |
| **1.00** | **0.7920** ✅ |
| 5.00 | 0.7822 |
| 10.00 | 0.7750 |

---

## Interpretation

The results show that the model achieved its highest average Macro F1-score (**0.7920**) when the regularisation parameter was **C = 1.0**.

As the value of **C** increased from **0.01** to **1.0**, the model's performance improved, indicating that reducing excessive regularisation allowed the classifier to better capture patterns in the training data.

However, increasing **C** beyond **1.0** led to a decline in Macro F1-score. This suggests that weaker regularisation resulted in a more complex model that generalised less effectively to unseen validation data.

Therefore, **C = 1.0** provides the best balance between underfitting and overfitting for this dataset.

---

## Conclusion

Based on the GridSearchCV results, **C = 1.0** was selected as the optimal hyperparameter for the Logistic Regression classifier.

The final Logistic Regression model will therefore be trained using:

```python
LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)
```

This configuration achieved the highest cross-validated Macro F1-score and will be used for the subsequent model evaluation on the held-out test dataset.

# Naives Bayes


## Workflow

```text
Training Data
       │
       ▼
Try α = 0.01
       │
       ▼
3-Fold Cross Validation
       │
       ▼
Calculate Macro F1
       │
       ▼
Repeat for α = 0.05
       │
       ▼
Repeat for α = 0.10
       │
       ▼
Repeat for α = 0.50
       │
       ▼
Repeat for α = 1.00
       │
       ▼
Select Best Alpha
       │
       ▼
Train Final Naive Bayes Model
```

### Hyperparameter Tuning for Multinomial Naive Bayes

#### Objective

The objective of this experiment is to identify the optimal value of the **Laplace smoothing parameter (α)** for the Multinomial Naive Bayes classifier using **GridSearchCV**.

The following candidate values of **α** are evaluated:

```python
[0.01, 0.05, 0.1, 0.5, 1.0]
```

The model performance is evaluated using **3-fold cross-validation**, with **Macro F1-score** as the evaluation metric.

---

## What is Laplace Smoothing?

Multinomial Naive Bayes calculates the probability of each word belonging to a particular sentiment class.

Sometimes, a word appearing in the test data may never have appeared in the training data for a specific class.

Without smoothing:

\[
P(\text{word}|\text{class}) = 0
\]

Since Naive Bayes multiplies probabilities together, a single zero probability causes the entire prediction probability to become zero.

Laplace smoothing prevents this problem by adding a small constant (**α**) to every word count.

Instead of assigning a probability of zero, every word receives a small non-zero probability.

---

## Understanding the Alpha (α) Parameter

The **α (alpha)** parameter controls the amount of smoothing applied.

| Alpha Value | Interpretation |
|-------------:|----------------|
| 0.01 | Very little smoothing |
| 0.05 | Small amount of smoothing |
| 0.10 | Moderate smoothing |
| 0.50 | Strong smoothing |
| 1.00 | Standard Laplace smoothing |

A very small alpha may cause the model to overfit the training data, whereas a very large alpha may oversmooth the probabilities and reduce classification performance.

GridSearchCV identifies the value that provides the best balance.

---

## Why use GridSearchCV?

GridSearchCV automatically:

1. Trains a Naive Bayes model for each alpha value.
2. Performs 3-fold cross-validation.
3. Computes the average Macro F1-score.
4. Selects the alpha producing the highest score.

---


In [14]:
# Candidate values for the Laplace smoothing parameter (alpha)
nb_param_grid = {
    "alpha": [0.01, 0.05, 0.1, 0.5, 1.0]
}

# Initialise GridSearchCV for hyperparameter tuning
nb_grid = GridSearchCV(

    # Multinomial Naive Bayes classifier
    MultinomialNB(),

    # Hyperparameter values to evaluate
    nb_param_grid,

    # Perform 3-fold cross-validation
    cv=3,

    # Select the model with the highest Macro F1-score
    scoring="f1_macro",

    # Use all available CPU cores
    n_jobs=-1,

    # Suppress detailed output
    verbose=0,
)

# Train the GridSearchCV model
nb_grid.fit(X_train_vec, y_train)

# Retrieve the best alpha value
best_nb_alpha = nb_grid.best_params_["alpha"]



In [15]:
# Display the best alpha and corresponding Macro F1-score
print(f"Best alpha = {best_nb_alpha} (CV F1 macro = {nb_grid.best_score_:.4f})")

# Display the score for every alpha value tested
print(
    f"All scores: "
    f"{ {str(p['alpha']): round(s, 4) for p, s in zip(nb_grid.cv_results_['params'], nb_grid.cv_results_['mean_test_score'])} }"
)

Best alpha = 0.01 (CV F1 macro = 0.7516)
All scores: {'0.01': np.float64(0.7516), '0.05': np.float64(0.7516), '0.1': np.float64(0.7516), '0.5': np.float64(0.7515), '1.0': np.float64(0.7512)}


# Hyperparameter Tuning Results

## GridSearchCV Results

| Alpha | Mean Macro F1-score |
|-------:|--------------------:|
| **0.01** | **0.7516** ✅ |
| 0.05 | 0.7516 |
| 0.10 | 0.7516 |
| 0.50 | 0.7515 |
| 1.00 | 0.7512 |

---

## Interpretation

GridSearchCV evaluated five candidate values of the Laplace smoothing parameter (α) using 3-fold cross-validation.

The highest Macro F1-score (**0.7516**) was obtained with **α = 0.01**.

The results indicate that only a very small amount of smoothing is required for this dataset. Increasing the smoothing parameter beyond 0.01 did not improve model performance and produced only negligible changes in the Macro F1-score.

The differences between all evaluated alpha values are extremely small (less than 0.05%), indicating that the Multinomial Naive Bayes classifier is relatively insensitive to the choice of alpha when trained on TF-IDF features.

Therefore, **α = 0.01** was selected as the optimal smoothing parameter for the final Multinomial Naive Bayes model.

## Phase 2: Training the Final Models with best Hyper parameters

### Objective

After finding the best hyperparameters using GridSearchCV, the next step is to train the final models.

Instead of trying different parameter values again, we now use the **best values** found during hyperparameter tuning:

- **Logistic Regression:** C = 1.0
- **Multinomial Naive Bayes:** α = 0.01

These models are trained using the **entire training dataset** so they can learn as much information as possible before being evaluated.

---

### Why do we train the models again?

During GridSearchCV, multiple models are trained with different hyperparameter values to determine which configuration performs best.

Once the best hyperparameters have been identified, a new model is trained using those optimal values and the complete training dataset. This final model is then tested on unseen data to evaluate how well it can generalise to new customer reviews.

---

### Model Evaluation

After training, the models make predictions on the test dataset.

The predictions are then compared with the actual sentiment labels using the following evaluation metrics:

- **Accuracy** – Overall percentage of correct predictions.
- **Precision** – How often the predicted sentiment is correct.
- **Recall** – How many actual reviews of a sentiment are correctly identified.
- **F1-score** – A balanced measure of Precision and Recall.
- **Macro F1-score** – The average F1-score across all sentiment classes, giving equal importance to Negative, Neutral, and Positive reviews.

These metrics provide a complete picture of how well each model performs on unseen data.

## Model Training Workflow

```text
Dataset
   │
   ▼
Train/Test Split
   │
   ▼
TF-IDF Vectorization
   │
   ▼
GridSearchCV
(Find the Best Hyperparameters)
   │
   ├─────────────────────────────┐
   ▼                             ▼
Logistic Regression         Multinomial Naive Bayes
Best C = 1.0                Best α = 0.01
   │                             │
   ▼                             ▼
Train Final Model          Train Final Model
   │                             │
   ▼                             ▼
Predict Test Data          Predict Test Data
   │                             │
   └──────────────┬──────────────┘
                  ▼
        Compare Predictions
        with Actual Labels
                  │
                  ▼
      Evaluate Model Performance
                  │
                  ▼
Accuracy • Precision • Recall • F1-score • Macro F1
```

In [16]:
print("\nTraining Final Models")
print("-" * 50)

# Logistic Regression

print(f"\nTraining Logistic Regression (C={best_lr_C})...")

# Create the final Logistic Regression model using the best C value
lr = LogisticRegression(
    max_iter=1000,
    C=best_lr_C,
    solver="lbfgs",
    random_state=42,
)

# Train the model
lr.fit(X_train_vec, y_train)

# Generate predictions on the test set
lr_preds: list[int] = list(lr.predict(X_test_vec))

# Calculate overall accuracy
lr_acc: float = accuracy_score(y_test, lr_preds)

print(f"\nLogistic Regression Results")
print(f"Accuracy : {lr_acc:.4f} ({lr_acc * 100:.1f}%)")

# Display Precision, Recall and F1-score for each sentiment class
print(
    classification_report(
        y_test,
        lr_preds,
        target_names=["negative", "neutral", "positive"],
        digits=4,
    )
)




Training Final Models
--------------------------------------------------

Training Logistic Regression (C=1.0)...

Logistic Regression Results
Accuracy : 0.8059 (80.6%)
              precision    recall  f1-score   support

    negative     0.8258    0.8507    0.8381     43655
     neutral     0.6863    0.6490    0.6671     32895
    positive     0.8726    0.8824    0.8775     41921

    accuracy                         0.8059    118471
   macro avg     0.7949    0.7940    0.7942    118471
weighted avg     0.8036    0.8059    0.8045    118471



In [17]:
# Multinomial Naive Bayes

print(f"\nTraining Naive Bayes (alpha={best_nb_alpha})...")

# Create the final Naive Bayes model using the best alpha value
nb = MultinomialNB(alpha=best_nb_alpha)

# Train the model
nb.fit(X_train_vec, y_train)

# Generate predictions on the test set
nb_preds: list[int] = list(nb.predict(X_test_vec))

# Calculate overall accuracy
nb_acc: float = accuracy_score(y_test, nb_preds)

print(f"\nNaive Bayes Results")
print(f"Accuracy : {nb_acc:.4f} ({nb_acc * 100:.1f}%)")

# Display Precision, Recall and F1-score for each sentiment class
print(
    classification_report(
        y_test,
        nb_preds,
        target_names=["negative", "neutral", "positive"],
        digits=4,
    )
)


Training Naive Bayes (alpha=0.01)...

Naive Bayes Results
Accuracy : 0.7586 (75.9%)
              precision    recall  f1-score   support

    negative     0.7815    0.7820    0.7817     43655
     neutral     0.6066    0.6456    0.6255     32895
    positive     0.8672    0.8229    0.8445     41921

    accuracy                         0.7586    118471
   macro avg     0.7518    0.7501    0.7506    118471
weighted avg     0.7632    0.7586    0.7605    118471



## RESULT

## Model Performance Analysis

## Logistic Regression

The Logistic Regression model achieved an **accuracy of 80.6%**, making it the best-performing classical machine learning model in this experiment.

The model performed particularly well on the **Positive** and **Negative** sentiment classes, achieving F1-scores of **0.8775** and **0.8381**, respectively. However, the **Neutral** class was more challenging to classify, with an F1-score of **0.6671**.

The overall **Macro F1-score** of **0.7942** indicates that the model maintained good performance across all three sentiment classes while treating each class equally.

---

## Multinomial Naive Bayes

The Multinomial Naive Bayes model achieved an **accuracy of 75.9%**.

Similar to Logistic Regression, the model classified **Positive** and **Negative** reviews reasonably well. However, its performance on the **Neutral** class was lower, resulting in an overall **Macro F1-score** of **0.7506**.

Although Naive Bayes produced competitive results, its overall performance was lower than Logistic Regression across all evaluation metrics.

---

## Comparison of the Two Models

| Metric | Logistic Regression | Naive Bayes |
|---------|--------------------:|------------:|
| Accuracy | **80.6%** | 75.9% |
| Macro F1-score | **0.7942** | 0.7506 |
| Weighted F1-score | **0.8045** | 0.7605 |

The results show that **Logistic Regression outperformed Multinomial Naive Bayes** on this dataset.

One possible reason is that Logistic Regression is better suited to high-dimensional TF-IDF feature representations, allowing it to learn more discriminative decision boundaries between the three sentiment classes. In contrast, Multinomial Naive Bayes assumes that features are conditionally independent, which may not fully capture the relationships between words in customer reviews.

---

## Key Observations

- Logistic Regression achieved the highest overall performance.
- Both models classified **Positive** reviews more accurately than **Neutral** reviews.
- The **Neutral** sentiment class was the most difficult to classify, indicating that neutral reviews often contain more ambiguous language.
- Logistic Regression will be selected as the baseline classical machine learning model for comparison with transformer-based models in the next stage of the project.

# Why were Logistic Regression and Naive Bayes used?

This project uses **Logistic Regression** and **Multinomial Naive Bayes** as baseline machine learning models for sentiment classification.

These algorithms are widely used in Natural Language Processing (NLP), especially for text classification tasks, because they are simple, efficient, and perform well when combined with TF-IDF features.

---

## 1. Logistic Regression

Logistic Regression is a supervised machine learning algorithm used for classification problems.

In this project, it learns the relationship between TF-IDF features and the sentiment labels (Negative, Neutral, and Positive).

### Why was Logistic Regression selected?

- It performs exceptionally well on high-dimensional sparse text data.
- It works effectively with TF-IDF feature vectors.
- It is computationally efficient, even for large datasets.
- It produces strong baseline results for sentiment analysis.
- It provides interpretable and reproducible results.

---

## 2. Multinomial Naive Bayes

Multinomial Naive Bayes is a probabilistic classifier designed for text classification.

It predicts the sentiment of a review by calculating the probability of each sentiment class based on the words present in the review.

### Why was Multinomial Naive Bayes selected?

- It is specifically designed for text data.
- It is fast to train and predict.
- It performs well on bag-of-words and TF-IDF representations.
- It requires very little computational power.
- It is widely used as a benchmark model in sentiment analysis.

---

## Why compare these two models?

Although both algorithms are commonly used for text classification, they learn in different ways.

| Logistic Regression | Multinomial Naive Bayes |
|---------------------|-------------------------|
| Learns decision boundaries between classes | Learns class probabilities based on word frequencies |
| Discriminative model | Generative probabilistic model |
| Usually achieves higher accuracy | Usually trains faster |
| Better at capturing complex relationships | Simpler and computationally efficient |

Comparing these two models helps determine which learning approach is more suitable for the sentiment analysis task.

---

## Why not use only one model?

Using multiple baseline models provides a fair comparison and demonstrates that the selected model was chosen based on experimental results rather than assumptions.

After evaluation:

- Logistic Regression achieved an Accuracy of **80.6%** and a Macro F1-score of **0.7942**.
- Multinomial Naive Bayes achieved an Accuracy of **75.9%** and a Macro F1-score of **0.7506**.

The results show that Logistic Regression performed better across all evaluation metrics and was therefore selected as the stronger classical machine learning baseline for this project.

```text
                     Original Dataset
                            │
                            ▼
                  Train/Test Split
                    /             \
                   /               \
                  ▼                 ▼
         Training Set          Test Set
              │              (Kept Unseen)
              ▼
  5-Fold Stratified Cross-Validation
              │
      ┌───────┴────────┐
      │                │
      ▼                ▼
  Train Model     Validate Model
      │                │
      └───────┬────────┘
              ▼
     Repeat for 5 Folds
              │
              ▼
 Calculate Mean Macro F1-score
   and Standard Deviation
              │
              ▼
 Select the Best Performing Model
              │
              ▼
 Final Evaluation on Test Set
```

In [18]:
print("\n5-Fold Stratified Cross-Validation")
print("-" * 50)
print("Using the training dataset for model validation\n")

# Create a 5-fold Stratified Cross-Validation object
# Stratification ensures each fold maintains the original class distribution
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Store the Macro F1-score for each fold
lr_cv_scores = []
nb_cv_scores = []

# Perform 5-fold cross-validation
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):

    # Split the training data into training and validation subsets
    X_tr = [X_train[i] for i in tr_idx]
    X_val = [X_train[i] for i in val_idx]

    y_tr = [y_train[i] for i in tr_idx]
    y_val = [y_train[i] for i in val_idx]

    # Transform the text using the fitted TF-IDF vectorizer
    X_tr_v = vectorizer.transform(X_tr)
    X_val_v = vectorizer.transform(X_val)

    # Train Logistic Regression on the current training fold
    lr_fold = LogisticRegression(
        C=best_lr_C,
        max_iter=1000,
        solver="lbfgs",
        random_state=42
    )

    lr_fold.fit(X_tr_v, y_tr)

    # Evaluate Logistic Regression on the validation fold
    lr_f1 = f1_score(
        y_val,
        lr_fold.predict(X_val_v),
        average="macro",
        zero_division=0
    )

    lr_cv_scores.append(lr_f1)

    # Train Multinomial Naive Bayes on the current training fold
    nb_fold = MultinomialNB(alpha=best_nb_alpha)

    nb_fold.fit(X_tr_v, y_tr)

    # Evaluate Naive Bayes on the validation fold
    nb_f1 = f1_score(
        y_val,
        nb_fold.predict(X_val_v),
        average="macro",
        zero_division=0
    )

    nb_cv_scores.append(nb_f1)

    # Display the Macro F1-score for the current fold
    print(f"Fold {fold}/5 | LR Macro F1 = {lr_f1:.4f} | NB Macro F1 = {nb_f1:.4f}")




5-Fold Stratified Cross-Validation
--------------------------------------------------
Using the training dataset for model validation

Fold 1/5 | LR Macro F1 = 0.7914 | NB Macro F1 = 0.7506
Fold 2/5 | LR Macro F1 = 0.7945 | NB Macro F1 = 0.7516
Fold 3/5 | LR Macro F1 = 0.7943 | NB Macro F1 = 0.7534
Fold 4/5 | LR Macro F1 = 0.7933 | NB Macro F1 = 0.7538
Fold 5/5 | LR Macro F1 = 0.7925 | NB Macro F1 = 0.7515


In [19]:
# Calculate the average and standard deviation across all folds
lr_cv_mean = np.mean(lr_cv_scores)
lr_cv_std = np.std(lr_cv_scores)

nb_cv_mean = np.mean(nb_cv_scores)
nb_cv_std = np.std(nb_cv_scores)

# Display the overall cross-validation performance
print("\nCross-Validation Summary")
print(f"Logistic Regression : {lr_cv_mean:.4f} ± {lr_cv_std:.4f}")
print(f"Naive Bayes         : {nb_cv_mean:.4f} ± {nb_cv_std:.4f}")


Cross-Validation Summary
Logistic Regression : 0.7932 ± 0.0012
Naive Bayes         : 0.7522 ± 0.0012


## Cross-Validation Results

Five-fold Stratified Cross-Validation was performed on the training dataset to evaluate the consistency and generalisation ability of both machine learning models.

## Logistic Regression

The Logistic Regression model achieved an average **Macro F1-score of 0.7932 ± 0.0012**.

The Macro F1-score remained highly consistent across all five folds, ranging from **0.7914 to 0.7945**. The very small standard deviation indicates that the model produced stable performance regardless of how the training data was partitioned.

---

## Multinomial Naive Bayes

The Multinomial Naive Bayes model achieved an average **Macro F1-score of 0.7522 ± 0.0012**.

Although the model also demonstrated consistent performance across all folds, its average Macro F1-score was lower than that of Logistic Regression, indicating comparatively weaker classification performance.

---

## Comparison

| Model | Mean Macro F1 | Standard Deviation |
|--------|--------------:|-------------------:|
| Logistic Regression | **0.7932** | **± 0.0012** |
| Multinomial Naive Bayes | 0.7522 | ± 0.0012 |

The results show that **Logistic Regression consistently outperformed Multinomial Naive Bayes** across all five validation folds.

Both models exhibited very low standard deviations, indicating that their performance remained stable across different training-validation splits. However, Logistic Regression achieved a substantially higher average Macro F1-score, making it the stronger baseline model for this sentiment classification task.

## Save Artifacts

In [20]:
# ── Save artefacts ────────────────────────────────────────────────────────────
os.makedirs("models/classical", exist_ok=True)

with open("models/classical/tfidf_vectorizer.pkl",    "wb") as f:
    pickle.dump(vectorizer, f)
with open("models/classical/logistic_regression.pkl", "wb") as f:
    pickle.dump(lr, f)
with open("models/classical/naive_bayes.pkl",         "wb") as f:
    pickle.dump(nb, f)

print("\n✅  Models saved → models/classical/")


✅  Models saved → models/classical/


## Test

In [21]:
sample = "The delivery was quick and the product quality is excellent."

# Transform the review into TF-IDF features
sample_vec = vectorizer.transform([sample])

# Predict the sentiment
prediction = lr.predict(sample_vec)[0]

# Predict class probabilities
probabilities = lr.predict_proba(sample_vec)[0]

print(prediction)
print(probabilities)

2
[0.00315999 0.05933761 0.9375024 ]


In [22]:
label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

print("Prediction :", label_map[prediction])

for label, score in zip(label_map.values(), probabilities):
    print(f"{label}: {score:.4f}")

Prediction : positive
negative: 0.0032
neutral: 0.0593
positive: 0.9375


In [23]:
# Example review
review = "The delivery was fast and the product quality is excellent."

# Convert the review into TF-IDF features
review_vec = vectorizer.transform([review])

# Logistic Regression prediction
lr_prediction = lr.predict(review_vec)[0]
lr_probabilities = lr.predict_proba(review_vec)[0]

# Naive Bayes prediction
nb_prediction = nb.predict(review_vec)[0]
nb_probabilities = nb.predict_proba(review_vec)[0]

# Map numeric labels to sentiment names
label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

print("=" * 60)
print("Input Review:")
print(review)

print("\nLogistic Regression")
print("-------------------")
print("Prediction :", label_map[lr_prediction])
print(f"Confidence : {max(lr_probabilities):.4f}")
print("Class Probabilities:")
for label, prob in zip(label_map.values(), lr_probabilities):
    print(f"  {label:<8}: {prob:.4f}")

print("\nNaive Bayes")
print("-----------")
print("Prediction :", label_map[nb_prediction])
print(f"Confidence : {max(nb_probabilities):.4f}")
print("Class Probabilities:")
for label, prob in zip(label_map.values(), nb_probabilities):
    print(f"  {label:<8}: {prob:.4f}")

Input Review:
The delivery was fast and the product quality is excellent.

Logistic Regression
-------------------
Prediction : positive
Confidence : 0.9460
Class Probabilities:
  negative: 0.0018
  neutral : 0.0522
  positive: 0.9460

Naive Bayes
-----------
Prediction : positive
Confidence : 0.7407
Class Probabilities:
  negative: 0.0582
  neutral : 0.2011
  positive: 0.7407


## Inference Demonstration on Unseen Review

To verify that the trained models can generalise to new, unseen data, a manually written customer review was provided as input. Both the Logistic Regression and Multinomial Naive Bayes models correctly classified the review as **positive**.

### Input Review

> *"The delivery was fast and the product quality is excellent."*

### Prediction Results

| Model | Predicted Sentiment | Confidence |
|-------|---------------------|-----------:|
| Logistic Regression | Positive | **94.60%** |
| Multinomial Naive Bayes | Positive | **74.07%** |

### Class Probabilities

#### Logistic Regression

| Sentiment | Probability |
|-----------|------------:|
| Negative | 0.0018 |
| Neutral | 0.0522 |
| Positive | **0.9460** |

#### Multinomial Naive Bayes

| Sentiment | Probability |
|-----------|------------:|
| Negative | 0.0582 |
| Neutral | 0.2011 |
| Positive | **0.7407** |

### Discussion

Both models correctly identified the review as **positive**, indicating that they successfully recognised the positive sentiment expressed through words such as *fast*, *quality*, and *excellent*.

The Logistic Regression model assigned a confidence score of **94.60%**, demonstrating a strong distinction between the positive and the other sentiment classes. In contrast, the Multinomial Naive Bayes model predicted the same class with a lower confidence of **74.07%**, allocating a larger probability to the neutral class. This suggests that Naive Bayes was comparatively more uncertain when making the prediction.

These observations are consistent with the quantitative evaluation results obtained on the test dataset, where Logistic Regression achieved higher accuracy and Macro F1-score than Multinomial Naive Bayes. The inference example therefore provides additional qualitative evidence that Logistic Regression is the more reliable classical machine learning model for sentiment classification in the FeedbackIQ system.

In [24]:
reviews = [
    "Excellent quality and very fast delivery.",                 # Positive
    "The package arrived yesterday.",                            # Neutral
    "Terrible product. Waste of money.",                         # Negative
    "Customer support solved my issue quickly.",                 # Positive
    "The colour is slightly different from the picture.",        # Neutral
    "I will never buy this product again.",                      # Negative
]

label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

for review in reviews:

    vec = vectorizer.transform([review])

    lr_pred = label_map[lr.predict(vec)[0]]
    nb_pred = label_map[nb.predict(vec)[0]]

    print("-" * 70)
    print("Review:", review)
    print("Logistic Regression:", lr_pred)
    print("Naive Bayes        :", nb_pred)

----------------------------------------------------------------------
Review: Excellent quality and very fast delivery.
Logistic Regression: positive
Naive Bayes        : positive
----------------------------------------------------------------------
Review: The package arrived yesterday.
Logistic Regression: negative
Naive Bayes        : negative
----------------------------------------------------------------------
Review: Terrible product. Waste of money.
Logistic Regression: negative
Naive Bayes        : negative
----------------------------------------------------------------------
Review: Customer support solved my issue quickly.
Logistic Regression: positive
Naive Bayes        : negative
----------------------------------------------------------------------
Review: The colour is slightly different from the picture.
Logistic Regression: negative
Naive Bayes        : neutral
----------------------------------------------------------------------
Review: I will never buy this produ

# MLFLOW

In [25]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, f1_score


# Logistic Regression Experiment


with mlflow.start_run(run_name="Logistic Regression - First Experiment"):

    # Log hyperparameters
    mlflow.log_params({
        "algorithm": "Logistic Regression",
        "vectorizer": "TF-IDF",
        "C": best_lr_C,
        "max_iter": 1000
    })

    # Log evaluation metrics
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, lr_preds),
        "macro_f1": f1_score(y_test, lr_preds, average="macro"),
        "weighted_f1": f1_score(y_test, lr_preds, average="weighted")
    })

    # Log trained model
    mlflow.sklearn.log_model(
        sk_model=lr,
        name="logistic_regression_model"
    )

print("✅ Logistic Regression logged successfully!")


# Naive Bayes Experiment

with mlflow.start_run(run_name="Naive Bayes - First Experiment"):

    # Log hyperparameters
    mlflow.log_params({
        "algorithm": "Multinomial Naive Bayes",
        "vectorizer": "TF-IDF",
        "alpha": best_nb_alpha
    })

    # Log evaluation metrics
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, nb_preds),
        "macro_f1": f1_score(y_test, nb_preds, average="macro"),
        "weighted_f1": f1_score(y_test, nb_preds, average="weighted")
    })

    # Log trained model
    mlflow.sklearn.log_model(
        sk_model=nb,
        name="naive_bayes_model"
    )

print("✅ Naive Bayes logged successfully!")

✅ Logistic Regression logged successfully!
✅ Naive Bayes logged successfully!


In [26]:
import mlflow
print(mlflow.__version__)
print(mlflow.__file__)

3.14.0
/user/HS400/ms05193/Documents/Dataset/venv/lib/python3.12/site-packages/mlflow/__init__.py


In [27]:
import sys
print(sys.executable)

/user/HS400/ms05193/Documents/Dataset/venv/bin/python


In [ ]:
import sys

!{sys.executable} -m mlflow ui --host 127.0.0.1 --port 5000

Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/07/22 18:25:05 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/07/22 18:25:05 INFO:     Started parent process [2759743]
/user/HS400/ms05193/Documents/Dataset/venv/lib/python3.12/site-packages/mlflow/server/fastapi_app.py:17: StarletteDeprecationWarning: starlette.middleware.wsgi is deprecated and will be removed in a future release. Please refer to https://github.com/abersheeran/a2wsgi as a replacement.
  from starlette.middleware.wsgi import WSGIResponder, build_environ
/user/HS400/ms05193/Documents/Dataset/venv/lib/python3.12/site-packages/mlflow/server/fastapi_app.py:17: StarletteDeprecationWarning: starlette.middleware.wsgi is deprecated and 

## Set Experiments

In [ ]:
mlflow.set_experiment("FeedbackIQ Classical Models")

In [ ]:
with mlflow.start_run():

    mlflow.sklearn.log_model(
        sk_model=lr,
        name="lr_model",
        registered_model_name="FeedbackIQ-LogisticRegression"
    )

In [ ]:
with mlflow.start_run():

    mlflow.sklearn.log_model(
        sk_model=nb,
        name="nb_model",
        registered_model_name="FeedbackIQ-NaiveBayes"
    )